In [ ]:
!pip install pandas



zsh:1: command not found: pip


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Creamos un Dataset simulado con datos numéricos y categóricos mezclados
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'edad': np.random.randint(18, 70, n),
    'ingreso': np.random.randint(20000, 120000, n),
    'ciudad': np.random.choice(['CDMX', 'Guadalajara', 'Monterrey'], n),
    'compro': np.random.choice([0, 1], n, p=[0.7, 0.3]) # Variable objetivo
})

# Introducimos algunos valores nulos para simular la realidad
df.loc[df['edad'].sample(frac=0.1, random_state=42).index, 'edad'] = np.nan

X = df.drop(columns=['compro'])
y = df['compro']

# Separamos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Definimos las transformaciones para las columnas NUMÉRICAS
transformador_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Rellena nulos con la mediana
    ('scaler', StandardScaler())                   # Normaliza (media 0, varianza 1)
])

# 3. Definimos las transformaciones para las columnas CATEGÓRICAS
transformador_categorico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='desconocido')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Convierte texto en columnas binarias (0 y 1)
])

# 4. Unimos ambas estrategias usando ColumnTransformer
preprocesador = ColumnTransformer(
    transformers=[
        ('num', transformador_numerico, ['edad', 'ingreso']),
        ('cat', transformador_categorico, ['ciudad'])
    ])

# 5. Creamos el Pipeline maestro (Preprocesamiento + Modelo de Machine Learning)
modelo_pipeline = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('clasificador', LogisticRegression())
])

# --- Entrenamiento y Predicción con UNA sola línea ---
# Ajusta el preprocesador (calcula medias y categorías) y entrena la Regresión Logística
modelo_pipeline.fit(X_train, y_train)

# Predice sobre el set de prueba (los datos nuevos pasan automáticamente por la limpieza)
predicciones = modelo_pipeline.predict(X_test)

# Evaluamos el resultado
precision = accuracy_score(y_test, predicciones)
print(f"Precisión d el Pipeline en Test: {precision * 100:.2f}%")

Precisión del Pipeline en Test: 63.50%
